# Did removing the preset gift amount improve a donation checkout?

The decision is whether a checkout variant raises more money per visitor, not only whether it produces more donors. This notebook uses generated event records. It does not reproduce the assessment data or its exact results.

## 1. Generate a synthetic experiment

The generated variant increases donation probability and slightly lowers gift size. This creates a realistic trade-off: conversion can improve while revenue per visitor stays flat. These values are simulation settings, not findings from the assessment.

In [ ]:
import math
import random
from collections import defaultdict

random.seed(205)
VISITORS_PER_ARM = 6000
events = []

for group in ('control', 'variant'):
    for number in range(VISITORS_PER_ARM):
        visitor_id = f'SYN-{group}-{number:05d}'
        # A visitor can open the checkout more than once. Count the visitor once.
        view_count = random.choices([1, 2, 3], weights=[0.75, 0.20, 0.05])[0]
        for _ in range(view_count):
            events.append({'visitor_id': visitor_id, 'group': group, 'event': 'checkout_view', 'amount': 0.0})

        conversion_probability = 0.116 if group == 'control' else 0.126
        if random.random() < conversion_probability:
            recurring = random.random() < 0.12
            charge_count = random.randint(2, 3) if recurring else 1
            amount_scale = 1.0 if group == 'control' else 0.92
            for _ in range(charge_count):
                amount = random.lognormvariate(3.4, 0.9) * amount_scale
                events.append({'visitor_id': visitor_id, 'group': group, 'event': 'charge', 'amount': amount})

print(f'Synthetic visitors: {2 * VISITORS_PER_ARM:,}')
print(f'Synthetic event rows: {len(events):,}')

## 2. Build one outcome row per visitor

Use unique assigned visitors as the denominator. Sum each visitor's eligible charges. Non-donors remain in the analysis with zero revenue.

In [ ]:
visitor_revenue = defaultdict(float)
visitor_views = defaultdict(int)
visitor_group = {}
for row in events:
    visitor_id = row['visitor_id']
    visitor_group[visitor_id] = row['group']
    if row['event'] == 'checkout_view':
        visitor_views[visitor_id] += 1
    elif row['event'] == 'charge':
        visitor_revenue[visitor_id] += row['amount']

outcomes = {'control': [], 'variant': []}
for visitor_id, group in visitor_group.items():
    if visitor_views[visitor_id] > 0:
        outcomes[group].append(visitor_revenue[visitor_id])

for group in ('control', 'variant'):
    users = outcomes[group]
    donors = sum(value > 0 for value in users)
    total_revenue = sum(users)
    rpv = total_revenue / len(users)
    donor_rate = donors / len(users)
    avg_revenue_per_donor = total_revenue / donors if donors else 0.0
    print(f'{group}: visitors={len(users):,}, donors={donors:,}, conversion={donor_rate:.2%}, revenue/visitor=${rpv:.2f}, revenue/donor=${avg_revenue_per_donor:.2f}')

## 3. Test conversion and revenue per visitor

A two-proportion z-test compares conversion rates. For revenue, resample visitors within each arm and recompute the difference in mean revenue per visitor. The resampling unit matches the visitor-level assignment.


In [ ]:
control = outcomes['control']
variant = outcomes['variant']
n_c, n_v = len(control), len(variant)
x_c = sum(value > 0 for value in control)
x_v = sum(value > 0 for value in variant)
p_c, p_v = x_c / n_c, x_v / n_v
p_pool = (x_c + x_v) / (n_c + n_v)
se_conversion = math.sqrt(p_pool * (1 - p_pool) * (1 / n_c + 1 / n_v))
z = (p_v - p_c) / se_conversion
p_value_conversion = math.erfc(abs(z) / math.sqrt(2))
print(f'Conversion difference: {(p_v - p_c):.2%} (variant minus control)')
print(f'Two-sided p-value for conversion: {p_value_conversion:.4f}')

bootstrap_differences = []
for _ in range(1000):
    sample_c = [random.choice(control) for _ in control]
    sample_v = [random.choice(variant) for _ in variant]
    bootstrap_differences.append(sum(sample_v) / n_v - sum(sample_c) / n_c)
bootstrap_differences.sort()
low = bootstrap_differences[int(0.025 * len(bootstrap_differences))]
high = bootstrap_differences[int(0.975 * len(bootstrap_differences))]
rpv_difference = sum(variant) / n_v - sum(control) / n_c
print(f'Revenue-per-visitor difference: ${rpv_difference:.2f}')
print(f'Bootstrap 95% interval: (${low:.2f}, ${high:.2f})')

The interval describes only this synthetic sample. If it includes zero, the simulation does not establish a revenue-per-visitor lift, even if conversion improves. Do not use the event-level charge amount as a substitute for total revenue per assigned visitor.

## 4. Before deciding to ship

- Confirm that each visitor has one stable assignment.
- Count each eligible checkout visitor once.
- Set the revenue window and include zero-revenue visitors.
- Sum repeat and recurring charges at the visitor level.
- Check sample balance, missing assignments, refunds, and chargebacks.
- Predefine one primary metric and guardrails.
- Review revenue per visitor with conversion and gift-size diagnostics.

## Takeaway

A conversion lift is promising, but it does not by itself prove that the organization will raise more money. Test revenue per randomized visitor directly.